# 01 - Data Understanding and Exploratory Data Analysis

## Objective
This notebook aims to characterize the ECG dataset from both a general and experiment-oriented perspective.

The analysis has two purposes:
1. to describe the dataset structure, demographic composition, and signal properties;
2. to support methodological decisions for the privacy-risk experiments, namely preprocessing, lead selection, segmentation, and similarity analysis.

Because this work is part of a thesis experiment, the EDA is designed not only to describe the data, but also to justify the experimental pipeline.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from config import WFDB_RECORDS_DIR
from data_loading import load_record
from eda_utils import (
    add_afl_label,
    build_demographic_table,
    build_metadata_table,
    build_missing_summary,
    compute_intra_inter_similarity,
    plot_random_ecg,
    plot_similarity_distributions,
    zscore_signal,
)

wfdb_path = WFDB_RECORDS_DIR



## Dataset overview

The dataset used in this experiment is the PhysioNet database *A large-scale 12-lead electrocardiogram database for arrhythmia study*.

According to the dataset description, it contains tens of thousands of 12-lead ECG recordings, each with fixed duration and standard sampling frequency, making it suitable for large-scale signal analysis and downstream machine learning experiments. In the reference dissertation, the same database is described as containing ECG recordings stored in `.mat` and `.hea` files, with 1 record corresponding to 1 patient, signal duration of 10 seconds, and metadata such as age, sex, diagnosis codes, sampling frequency, and number of samples. 

In this thesis experiment, the goal is not arrhythmia classification itself, but privacy-oriented analysis of ECG signals, including re-identification and linkability risk.

In [ ]:
hea_files = list(wfdb_path.rglob("*.hea"))
mat_files = list(wfdb_path.rglob("*.mat"))

print("Total records:", len(hea_files))
print("MAT files:", len(mat_files))
print("Consistent:", len(hea_files) == len(mat_files))

## Dataset integrity

This step verifies whether the downloaded dataset is structurally complete. In WFDB-based ECG datasets, each recording is expected to have a matching header file (`.hea`) and signal file (`.mat` or equivalent). Ensuring this correspondence is important before any loading, preprocessing, or segmentation step.

In [ ]:
record = hea_files[0].with_suffix("")
signal, fs, leads = load_record(record)

print("Record ID:", hea_files[0].stem)
print("Shape:", signal.shape)
print("Sampling rate:", fs)
print("Duration (s):", signal.shape[0] / fs)
print("Leads:", leads)
print("Min amplitude:", np.min(signal))
print("Max amplitude:", np.max(signal))

## Single-record inspection

A first inspection of one ECG recording is used to confirm the empirical characteristics of the dataset:
- number of samples,
- number of leads,
- sampling frequency,
- signal duration,
- amplitude range.

This step is important to validate that the signals match the documented structure before any general analysis is performed.

In [ ]:
fig, axes = plt.subplots(12,1, figsize=(14,18), sharex=True)

for i in range(12):
    axes[i].plot(signal[:,i])
    axes[i].set_title(leads[i])
    axes[i].set_ylabel("Amp.")

plt.xlabel("Samples")
plt.tight_layout()
plt.show()

## Visual inspection of ECG morphology

Plotting all leads of one recording allows a first visual assessment of:
- signal morphology,
- lead-specific differences,
- possible artifacts or abnormal patterns.

This visual step is especially relevant in ECG analysis, where lead orientation and waveform morphology carry important information.

In [ ]:
metadata_df, metadata_errors_df = build_metadata_table(hea_files, load_record)

print(f"Metadata rows: {len(metadata_df)}")
print(f"Metadata loading errors: {len(metadata_errors_df)}")
metadata_df.head()

In [ ]:
metadata_df.describe()

In [ ]:
metadata_df.isna().sum()

## Metadata summary table

A metadata table is built to summarize structural and statistical properties of the ECG recordings. This allows:
- checking consistency across records,
- identifying corrupted or anomalous files,
- quantifying basic amplitude variation,
- supporting downstream EDA steps.

In [ ]:
print(metadata_df["samples"].value_counts().sort_index())
print(metadata_df["n_leads"].value_counts().sort_index())
print(metadata_df["fs"].value_counts().sort_index())
print(metadata_df["duration_sec"].describe())

## Structural consistency

This analysis checks whether the ECG recordings are structurally homogeneous in terms of:
- number of samples,
- number of leads,
- sampling rate,
- duration.

Structural homogeneity is important because later stages such as segmentation, feature extraction, and similarity analysis assume comparable signal format across patients.

In [ ]:
demo_df, demo_errors_df = build_demographic_table(hea_files)

print(f"Demographic rows: {len(demo_df)}")
print(f"Demographic parsing errors: {len(demo_errors_df)}")
demo_df.head()

In [ ]:
demo_df.isna().sum()

## Demographic and clinical metadata

A second metadata table is extracted from the WFDB header files. It contains demographic and clinical information such as age, sex, and diagnosis codes.

In [ ]:
demo_df["sex"].value_counts(dropna=False)

In [ ]:
demo_df["sex"].value_counts(dropna=False).plot(kind="bar", figsize=(6,4))
plt.title("Sex distribution")
plt.ylabel("Number of records")
plt.xlabel("Sex")
plt.show()

In [ ]:
demo_df = add_afl_label(demo_df, afl_code="164889003")

demo_df["label"].value_counts()



In [ ]:
demo_df["label"].value_counts().plot(kind="bar")

plt.title("Distribution of AFL vs Non-AFL")
plt.xlabel("Class")
plt.ylabel("Number of records")
plt.show()

In [ ]:
pd.crosstab(demo_df["sex"], demo_df["label"])

In [ ]:
pd.crosstab(demo_df["sex"], demo_df["label"]).plot(
    kind="bar",
    figsize=(7,4)
)

plt.title("AFL vs Non-AFL by Sex")
plt.xlabel("Sex")
plt.ylabel("Number of records")
plt.show()

In [ ]:
sex_class_pct = pd.crosstab(
    demo_df["sex"],
    demo_df["label"],
    normalize="index"
)

sex_class_pct

In [ ]:
sex_class_pct.plot(kind="bar", stacked=True)

plt.title("Proportion of AFL by Sex")
plt.ylabel("Proportion")
plt.show()

In [ ]:
ct = pd.crosstab(demo_df["sex"], demo_df["label"])

ct.plot(
    kind="bar",
    stacked=True,
    figsize=(7,4)
)

plt.title("AFL vs Non-AFL distribution by sex")
plt.xlabel("Sex")
plt.ylabel("Number of records")
plt.show()

## Class distribution and demographic breakdown

The dataset was further analyzed to determine the distribution of atrial flutter (AFL) and non-AFL recordings. The results show a strong class imbalance, with AFL representing a small fraction of the total dataset.

This imbalance is expected in clinical datasets and has implications for downstream modeling and evaluation.

Additionally, the distribution of AFL and non-AFL cases was analyzed across sex categories to investigate potential demographic differences in the dataset population.

In [ ]:
ages = pd.to_numeric(demo_df["age"], errors="coerce")

plt.figure(figsize=(7,4))
plt.hist(ages.dropna(), bins=30)
plt.title("Age distribution")
plt.xlabel("Age")
plt.ylabel("Frequency")
plt.show()

## Demographic analysis

Demographic analysis is relevant because ECG characteristics may vary with age and sex. This step helps characterize the population represented in the dataset and identify possible sources of physiological variability.

In [ ]:
plt.figure(figsize=(7,4))
plt.hist(metadata_df["global_mean"], bins=50)
plt.title("Distribution of signal mean values")
plt.xlabel("Mean amplitude")
plt.ylabel("Frequency")
plt.show()



In [ ]:
plt.figure(figsize=(7,4))
plt.hist(metadata_df["global_std"], bins=50)
plt.title("Distribution of signal standard deviation")
plt.xlabel("Standard deviation")
plt.ylabel("Frequency")
plt.show()



In [ ]:
plt.figure(figsize=(7,4))
plt.hist(metadata_df["global_min"], bins=50)
plt.title("Distribution of minimum amplitude")
plt.xlabel("Minimum amplitude")
plt.ylabel("Frequency")
plt.show()



In [ ]:
plt.figure(figsize=(7,4))
plt.hist(metadata_df["global_max"], bins=50)
plt.title("Distribution of maximum amplitude")
plt.xlabel("Maximum amplitude")
plt.ylabel("Frequency")
plt.show()



## Statistical signal analysis

These distributions provide a general view of ECG amplitude behavior across records and help identify possible outliers or scaling issues.

In [ ]:
plot_random_ecg(hea_files, load_record, n=5, lead_idx=1)

In [ ]:
plt.figure(figsize=(12,4))

for i in range(12):
    plt.plot(signal[:,i], label=leads[i])

plt.legend()
plt.title("Comparison between ECG leads")
plt.show()

## Inter-record variability

Plotting the same lead across multiple recordings provides a first qualitative view of inter-subject variability. This is relevant for later privacy-oriented experiments, where subject-related signal structure is important.

In [ ]:
corr = np.corrcoef(signal.T)

plt.figure(figsize=(8,6))
plt.imshow(corr, cmap="coolwarm")
plt.colorbar()
plt.title("Correlation between ECG leads")
plt.xticks(range(len(leads)), leads, rotation=45)
plt.yticks(range(len(leads)), leads)
plt.tight_layout()
plt.show()

## Lead correlation analysis

This analysis evaluates the level of redundancy or complementarity across ECG leads. Strong correlations suggest overlapping information, while lower correlations indicate that some leads may provide additional signal perspectives.

This is relevant for decisions such as using a single lead or selecting a subset of leads.

In [ ]:
from sklearn.decomposition import PCA

segments = []
record_ids = []
expected_length = None

for hea in hea_files[:200]:
    record_path = hea.with_suffix("")
    signal, fs, leads = load_record(record_path)

    lead_signal = zscore_signal(signal[:, 1])

    if expected_length is None:
        expected_length = len(lead_signal)

    if len(lead_signal) != expected_length:
        continue

    segments.append(lead_signal)
    record_ids.append(hea.stem)

X = np.array(segments)

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X)

plt.figure(figsize=(6,6))
plt.scatter(X_pca[:, 0], X_pca[:, 1], alpha=0.7)
plt.title("PCA projection of normalized lead II signals")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()

print(f"Signals used in PCA: {len(record_ids)}")



## PCA visualization

PCA is used as an exploratory tool to assess whether ECG recordings exhibit visible structure or variability in a reduced space.

In [ ]:
similarity_df, similarity_errors_df = compute_intra_inter_similarity(
    records=hea_files,
    loader=load_record,
    n_records=100,
    lead_idx=1,
    window_sec=2.0,
    step_sec=1.0,
    random_state=42,
    normalize=True,
)

print(f"Similarity rows: {len(similarity_df)}")
print(f"Similarity loading errors: {len(similarity_errors_df)}")
similarity_df.groupby("pair_type")["similarity"].describe()



In [ ]:
plot_similarity_distributions(similarity_df)

In [ ]:
missing_summaries = build_missing_summary({
    "metadata_df": metadata_df,
    "demo_df": demo_df,
})

for name, summary in missing_summaries.items():
    print(f"\n=== {name} ===")
    print(f"Total rows: {len(locals()[name])}")
    print(summary[summary["missing_count"] > 0] if (summary["missing_count"] > 0).any() else "No missing values.")

print("\n=== parsing summary ===")
print(f"Metadata errors: {len(metadata_errors_df)}")
print(f"Demographic errors: {len(demo_errors_df)}")
print(f"Similarity errors: {len(similarity_errors_df)}")



## Similarity analysis for privacy-oriented experimentation

This analysis compares ECG segments from the same record and from different records. The purpose is to assess whether the signal contains subject-related structure that may later support re-identification or linkability risk analysis.

## Implications for the experimental design

The previous analyses support several methodological decisions for the next stages of the experiment:

- whether one lead may be sufficient for a baseline setup;
- whether normalization or denoising is likely to be needed;
- whether segmentation into shorter windows is feasible;
- whether the ECG signal exhibits subject-related structure relevant for privacy-risk experiments.

Because this work is part of a thesis experiment, the EDA is used not only to describe the dataset, but also to justify the design of the next steps.

## Conclusions

This notebook provided a structured data understanding and exploratory analysis of the ECG dataset, combining:
- dataset characterization,
- demographic analysis,
- signal inspection,
- statistical analysis,
- and privacy-oriented exploratory analysis.

These results will support the design of the preprocessing, segmentation, and feature extraction stages of the thesis experiment.